# ANALISA TIMBANG — Custom CNN Training (Colab)

End-to-end training untuk model OCR 7-segment timbangan. Run cells in order pada Google Colab dengan **T4 GPU**.

**Prerequisite:**
- Punya ZIP file dari POC PWA (`analisa-training-YYYY-MM-DD.zip`)
- Min. 500 sample (rekomendasi 1000+)

**Output:** `model.onnx` siap deploy ke POC.

## Step 1: Setup Environment

In [ ]:
# Verify GPU
!nvidia-smi

In [ ]:
# Clone repo (adjust URL ke repo Anda)
# !git clone https://github.com/your-org/mobile-ocr.git
# %cd mobile-ocr/training

# OR upload semua .py files manually via Colab file browser kiri,
# lalu jalankan dari /content/

%cd /content/training

In [ ]:
!pip install -q -r requirements.txt

## Step 2: Upload Dataset ZIP

In [ ]:
# Option A: Upload via browser
from google.colab import files
uploaded = files.upload()  # pilih file analisa-training-*.zip
import shutil
for name in uploaded:
    shutil.move(name, f'data/{name}')
    print(f'Saved: data/{name}')

# Option B: Mount Google Drive jika file di sana
# from google.colab import drive
# drive.mount('/content/drive')
# !cp '/content/drive/MyDrive/path/analisa-training-2026-05-15.zip' data/

## Step 3: Prepare Dataset

In [ ]:
import glob
zip_files = glob.glob('data/analisa-training-*.zip')
assert zip_files, 'Tidak ada file ZIP di data/'
zip_path = zip_files[0]
print(f'Using: {zip_path}')

!python prepare.py --input {zip_path} --output data/prepared/ --clean

In [ ]:
# Visualize beberapa sample untuk sanity check
import os, random, cv2
import matplotlib.pyplot as plt
import csv

with open('data/prepared/train/labels.csv') as f:
    rows = list(csv.DictReader(f))

samples = random.sample(rows, min(12, len(rows)))
fig, axes = plt.subplots(3, 4, figsize=(14, 6))
for ax, row in zip(axes.flat, samples):
    img = cv2.imread(f'data/prepared/train/images/{row["filename"]}', cv2.IMREAD_GRAYSCALE)
    ax.imshow(img, cmap='gray')
    ax.set_title(f'"{row["ground_truth"]}"', fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Step 4: Train Model

In [ ]:
# Default config: 50 epochs, batch 32, lr 1e-3
# Adjust sesuai dataset size:
#   < 500 samples  → epochs 30, batch 16
#   500-2000       → epochs 50, batch 32 (default)
#   > 2000         → epochs 80, batch 64

!python train.py --data data/prepared/ --output models/cnn_v1/ --epochs 50 --batch 32

In [ ]:
# Show training curves
from PIL import Image
import matplotlib.pyplot as plt
img = Image.open('models/cnn_v1/training_curves.png')
plt.figure(figsize=(14, 4))
plt.imshow(img)
plt.axis('off')
plt.show()

## Step 5: Evaluate on Test Set

In [ ]:
!python evaluate.py --model models/cnn_v1/best.h5 --data data/prepared/test/

In [ ]:
# Show confusion matrix
from PIL import Image
import matplotlib.pyplot as plt
img = Image.open('models/cnn_v1/confusion_matrix.png')
plt.figure(figsize=(8, 6))
plt.imshow(img)
plt.axis('off')
plt.show()

## Step 6: Export ONNX

In [ ]:
!python export_onnx.py --model models/cnn_v1/best.h5 --output models/cnn_v1/model.onnx

## Step 7: Download Model

In [ ]:
from google.colab import files
# Download ONNX untuk deploy ke POC
files.download('models/cnn_v1/model.onnx')
files.download('models/cnn_v1/evaluation_report.json')
files.download('models/cnn_v1/training_curves.png')
files.download('models/cnn_v1/confusion_matrix.png')

## Next Steps di POC PWA

1. Copy `model.onnx` ke `poc/public/models/cnn_v1.onnx`
2. `npm install onnxruntime-web` di POC folder
3. Update `poc/src/lib/ocr.ts` — replace Tesseract dengan ONNX inference
4. Test akurasi di field, validate ≥95% accuracy
5. Promote ke production (MVP Phase 1)